# Gold Layer — Quality KPI Aggregations
OEE, defect rates, equipment health, shift performance, product rankings, weekly trends, and line scorecards.

In [ ]:
spark.conf.set('spark.sql.parquet.vorder.default', 'true')
spark.conf.set('spark.databricks.delta.optimizeWrite.enabled', 'true')
spark.conf.set('spark.databricks.delta.optimizeWrite.binSize', '1g')

In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, avg as _avg, count, countDistinct,
    min as _min, max as _max, current_timestamp,
    round as _round, when, lit, dense_rank,
    weekofyear, month, dayofweek, lag, stddev
)
from pyspark.sql.window import Window

In [ ]:
# === TABLE 1: Daily Production Summary ===
df_batches = spark.read.format('delta').table('silver_production_batches')

gold_production = (
    df_batches
    .groupBy('production_date', 'production_line', 'shift', 'product')
    .agg(
        count('*').alias('batch_count'),
        _sum('units_produced').alias('total_units_produced'),
        _sum('defect_count').alias('total_defects'),
        _sum('planned_units').alias('total_planned_units'),
        _sum('downtime_minutes').alias('total_downtime_minutes'),
        _avg('yield_pct').alias('avg_yield_pct'),
        _avg('defect_rate').alias('avg_defect_rate'),
    )
    .withColumn('availability', _round(1 - col('total_downtime_minutes') / (col('batch_count') * 480), 4))
    .withColumn('performance', _round(col('total_units_produced') / col('total_planned_units'), 4))
    .withColumn('quality', _round((col('total_units_produced') - col('total_defects')) / col('total_units_produced'), 4))
    .withColumn('oee', _round(col('availability') * col('performance') * col('quality') * 100, 2))
    .withColumn('production_week', weekofyear(col('production_date').cast('date')))
    .withColumn('production_month', month(col('production_date').cast('date')))
    .withColumn('day_of_week', dayofweek(col('production_date').cast('date')))
    .withColumn('oee_class',
                when(col('oee') >= 85, 'World Class')
                .when(col('oee') >= 60, 'Acceptable')
                .when(col('oee') >= 40, 'Needs Improvement')
                .otherwise('Critical'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_production.write.mode('overwrite').format('delta').saveAsTable('gold_production_daily_summary')
print(f'Gold production daily summary: {gold_production.count()} rows')

In [ ]:
# === TABLE 2: Equipment Health Daily ===
df_sensors = spark.read.format('delta').table('silver_sensor_readings')

gold_equipment = (
    df_sensors
    .groupBy('reading_date', 'machine_id', 'production_line', 'shift')
    .agg(
        count('*').alias('reading_count'),
        _avg('temperature').alias('avg_temperature'),
        _avg('pressure').alias('avg_pressure'),
        _avg('vibration').alias('avg_vibration'),
        _avg('humidity').alias('avg_humidity'),
        _sum(col('temp_anomaly').cast('int')).alias('temp_anomaly_count'),
        _sum(col('vibration_anomaly').cast('int')).alias('vibration_anomaly_count'),
        _min('temperature').alias('min_temperature'),
        _max('temperature').alias('max_temperature'),
        _min('vibration').alias('min_vibration'),
        _max('vibration').alias('max_vibration'),
        stddev('temperature').alias('temp_stddev'),
        stddev('vibration').alias('vibration_stddev'),
    )
    .withColumn('anomaly_rate',
                _round((col('temp_anomaly_count') + col('vibration_anomaly_count')) / col('reading_count') * 100, 2))
    .withColumn('temp_range', _round(col('max_temperature') - col('min_temperature'), 2))
    .withColumn('vibration_range', _round(col('max_vibration') - col('min_vibration'), 3))
    .withColumn('health_score',
                _round(when(100 - col('anomaly_rate') * 5 - col('temp_stddev') * 0.3 - col('vibration_stddev') * 2 > 0,
                            100 - col('anomaly_rate') * 5 - col('temp_stddev') * 0.3 - col('vibration_stddev') * 2)
                       .otherwise(0), 1))
    .withColumn('health_status',
                when(col('health_score') >= 90, 'Healthy')
                .when(col('health_score') >= 70, 'Warning')
                .when(col('health_score') >= 50, 'At Risk')
                .otherwise('Critical'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_equipment.write.mode('overwrite').format('delta').saveAsTable('gold_equipment_health_daily')
print(f'Gold equipment health daily: {gold_equipment.count()} rows')

In [ ]:
# === TABLE 3: Shift Performance Comparison ===
gold_shift = (
    df_batches
    .groupBy('shift', 'production_line')
    .agg(
        count('*').alias('total_batches'),
        _sum('units_produced').alias('total_units'),
        _sum('defect_count').alias('total_defects'),
        _avg('yield_pct').alias('avg_yield'),
        _avg('defect_rate').alias('avg_defect_rate'),
        _sum('downtime_minutes').alias('total_downtime'),
        _avg('downtime_minutes').alias('avg_downtime_per_batch'),
        countDistinct('production_date').alias('days_active'),
    )
    .withColumn('units_per_batch', _round(col('total_units') / col('total_batches'), 1))
    .withColumn('defects_per_batch', _round(col('total_defects') / col('total_batches'), 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_shift.write.mode('overwrite').format('delta').saveAsTable('gold_shift_performance')
print(f'Gold shift performance: {gold_shift.count()} rows')

In [ ]:
# === TABLE 4: Product Quality Rankings ===
w_rank = Window.orderBy(col('avg_yield').desc())

gold_product = (
    df_batches
    .groupBy('product')
    .agg(
        count('*').alias('total_batches'),
        _sum('units_produced').alias('total_units'),
        _sum('defect_count').alias('total_defects'),
        _avg('yield_pct').alias('avg_yield'),
        _avg('defect_rate').alias('avg_defect_rate'),
        _sum('downtime_minutes').alias('total_downtime'),
        countDistinct('production_line').alias('lines_used'),
    )
    .withColumn('quality_rank', dense_rank().over(w_rank))
    .withColumn('quality_tier',
                when(col('avg_yield') >= 98, 'Premium')
                .when(col('avg_yield') >= 95, 'Standard')
                .when(col('avg_yield') >= 90, 'Below Target')
                .otherwise('Reject Review'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_product.write.mode('overwrite').format('delta').saveAsTable('gold_product_quality')
print(f'Gold product quality: {gold_product.count()} rows')

In [ ]:
# === TABLE 5: Weekly Trends with WoW Change ===
weekly = (
    gold_production
    .groupBy('production_week', 'production_line')
    .agg(
        _sum('total_units_produced').alias('weekly_units'),
        _sum('total_defects').alias('weekly_defects'),
        _avg('oee').alias('weekly_avg_oee'),
        _sum('total_downtime_minutes').alias('weekly_downtime'),
        _sum('batch_count').alias('weekly_batches'),
    )
)

w_lag = Window.partitionBy('production_line').orderBy('production_week')
gold_weekly = (
    weekly
    .withColumn('prev_week_oee', lag('weekly_avg_oee').over(w_lag))
    .withColumn('oee_wow_change', _round(col('weekly_avg_oee') - col('prev_week_oee'), 2))
    .withColumn('prev_week_units', lag('weekly_units').over(w_lag))
    .withColumn('units_wow_pct',
                when(col('prev_week_units') > 0,
                     _round((col('weekly_units') - col('prev_week_units')) / col('prev_week_units') * 100, 1))
                .otherwise(lit(None)))
    .withColumn('trend_direction',
                when(col('oee_wow_change') > 2, 'Improving')
                .when(col('oee_wow_change') < -2, 'Declining')
                .otherwise('Stable'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_trends')
print(f'Gold weekly trends: {gold_weekly.count()} rows')

In [ ]:
# === TABLE 6: Line Scorecard (KPI cards) ===
gold_kpi = (
    df_batches
    .groupBy('production_line')
    .agg(
        _sum('units_produced').alias('total_units_all_time'),
        _sum('defect_count').alias('total_defects_all_time'),
        _sum('planned_units').alias('total_planned_all_time'),
        count('*').alias('total_batches_all_time'),
        _avg('yield_pct').alias('overall_yield'),
        _avg('defect_rate').alias('overall_defect_rate'),
        _sum('downtime_minutes').alias('total_downtime_all_time'),
        countDistinct('production_date').alias('operating_days'),
        countDistinct('product').alias('products_produced'),
    )
    .withColumn('units_per_day', _round(col('total_units_all_time') / col('operating_days'), 0))
    .withColumn('overall_oee', _round(
        (1 - col('total_downtime_all_time') / (col('total_batches_all_time') * 480)) *
        (col('total_units_all_time') / col('total_planned_all_time')) *
        ((col('total_units_all_time') - col('total_defects_all_time')) / col('total_units_all_time')) * 100, 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_kpi.write.mode('overwrite').format('delta').saveAsTable('gold_line_scorecard')
print(f'Gold line scorecard: {gold_kpi.count()} rows')

In [ ]:
# Optimize all Gold tables
for table in ['gold_production_daily_summary', 'gold_equipment_health_daily',
              'gold_shift_performance', 'gold_product_quality',
              'gold_weekly_trends', 'gold_line_scorecard']:
    spark.sql(f'OPTIMIZE {table}')
    cnt = spark.read.format('delta').table(table).count()
    print(f'{table}: {cnt} rows (optimized)')

print('\n=== Gold Layer Complete — 6 tables ===')